# Event field encoder
Shared by Models 03 and 06. The column groups and quantile boundaries are the original fitted preprocessing parameters.

In [ ]:
import hashlib
import math
def hashed(series, salt):
    values = series.fillna('').astype(str).str.strip().str.lower()
    uniques = values.unique()
    lookup = {value: 0 if not value else 1 + int.from_bytes(hashlib.blake2b((salt + ':' + value).encode(), digest_size=8).digest(), 'little') % 127 for value in uniques}
    return values.map(lookup).to_numpy(dtype=np.uint8)

## Convert one event into 115 discrete codes

In [ ]:
def encode_events(frame, meta):
    code = np.zeros((len(frame), 115), dtype=np.uint8)
    for i, (name, salt) in enumerate(zip(['event_id', 'event_id', 'user_id', 'user_id'], meta['id_salts'])):
        code[:, i] = hashed(frame[name], salt)
    dt = pd.to_datetime(frame.start_time, utc=True, errors='coerce', format='mixed')
    known = dt.notna()
    seconds = dt[known].astype('int64').to_numpy(dtype=np.float64) / 1e9
    code[known, 4] = 1 + np.searchsorted(meta['bins']['start_time'], seconds, side='right')
    for i, values in enumerate([dt.dt.month, dt.dt.dayofweek + 1, dt.dt.hour + 1], 5):
        code[:, i] = values.fillna(0)
    for i, name in enumerate(['city', 'state', 'zip', 'country'], 8):
        code[:, i] = hashed(frame[name], name)
    for i, (name, bound) in enumerate([('lat', 90), ('lng', 180)], 12):
        values = pd.to_numeric(frame[name], errors='coerce').to_numpy(dtype=np.float64)
        valid = np.isfinite(values) & (np.abs(values) <= bound)
        code[valid, i] = 1 + np.minimum(126, ((values[valid] + bound) / (2 * bound) * 127).astype(int))
    fields = [f'c_{j}' for group in meta['groups'] for j in group] + ['c_other']
    for i, name in enumerate(fields, 14):
        values = pd.to_numeric(frame[name], errors='coerce').to_numpy(dtype=np.float32)
        code[values == 0, i] = 1
        positive = np.isfinite(values) & (values > 0)
        code[positive, i] = 2 + np.searchsorted(meta['bins'][name], values[positive], side='right')
    return code

def encode_catalog(raw, meta, directory):
    total = meta['rows']
    codes = np.lib.format.open_memmap(directory / 'codes.npy', mode='w+', dtype='uint8', shape=(total, 115))
    ids = np.empty(total, dtype=np.uint64)
    offset = 0
    for frame in pd.read_csv(raw, chunksize=65536, dtype=str, keep_default_na=False):
        end = offset + len(frame)
        codes[offset:end] = encode_events(frame, meta)
        ids[offset:end] = frame.event_id.to_numpy(dtype=np.uint64)
        offset = end
    codes.flush()
    return codes, pd.Index(ids)

## Ten-token masked transformer

In [ ]:
def build_model(cards):
    import torch
    from torch import nn
    from torch.nn import functional as F
    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            self.n1 = nn.RMSNorm(128, eps=1e-6)
            self.n2 = nn.RMSNorm(128, eps=1e-6)
            self.qkv = nn.Linear(128, 384, bias=False)
            self.out = nn.Linear(128, 128, bias=False)
            self.gate = nn.Linear(128, 384, bias=False)
            self.up = nn.Linear(128, 384, bias=False)
            self.down = nn.Linear(384, 128, bias=False)
        def forward(self, x):
            qkv = self.qkv(self.n1(x)).reshape(x.shape[0], 10, 3, 4, 32).permute(2, 0, 3, 1, 4)
            attention = F.scaled_dot_product_attention(qkv[0], qkv[1], qkv[2], dropout_p=0.0, is_causal=False)
            x = x + self.out(attention.transpose(1, 2).reshape(x.shape[0], 10, 128))
            h = self.n2(x)
            return x + self.down(F.silu(self.gate(h)) * self.up(h))
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.heads = nn.ModuleList([nn.Embedding(sum(card), 128) for card in cards])
            self.position = nn.Parameter(torch.empty(10, 128))
            self.mask = nn.Parameter(torch.empty(128))
            self.blocks = nn.ModuleList([Block() for _ in range(4)])
            self.final_norm = nn.RMSNorm(128, eps=1e-6)
            for index, card in enumerate(cards):
                self.register_buffer(f'offset_{index}', torch.tensor(np.cumsum([0] + card[:-1]), dtype=torch.long), persistent=False)
            for parameter in self.parameters():
                nn.init.normal_(parameter, std=0.02) if parameter.ndim > 1 or parameter is self.mask else nn.init.ones_(parameter)
        def forward(self, fields, selected, residual_only=False):
            tokens = []
            targets = []
            start = 0
            for index, card in enumerate(cards):
                target = fields[:, start:start + len(card)]
                tokens.append(self.heads[index](target + getattr(self, f'offset_{index}')).sum(dim=1) / math.sqrt(len(card)))
                targets.append(target)
                start += len(card)
            x = torch.stack(tokens, dim=1)
            x = torch.where(selected[..., None], self.mask, x) + self.position
            for block in self.blocks:
                x = block(x)
            if residual_only:
                return x
            h = self.final_norm(x)
            losses = []
            for index, card in enumerate(cards):
                logits = F.linear(h[:, index], self.heads[index].weight)
                if len(set(card)) == 1:
                    loss = F.cross_entropy(logits.reshape(-1, card[0]), targets[index].reshape(-1), reduction='none').reshape(-1, len(card)).mean(dim=1)
                else:
                    offset = 0
                    pieces = []
                    for field, size in enumerate(card):
                        pieces.append(F.cross_entropy(logits[:, offset:offset + size], targets[index][:, field], reduction='none'))
                        offset += size
                    loss = torch.stack(pieces, dim=1).mean(dim=1)
                losses.append(loss)
            losses = torch.stack(losses, dim=1)
            return x, (losses * selected).sum(dim=1).div(3).mean(), losses
    return Model()

## Read the final residual before the output normalization

In [ ]:
@torch.inference_mode()
def residual_batches(model, codes):
    model.eval()
    for start in range(0, len(codes), 1024):
        fields = torch.from_numpy(np.array(codes[start:start + 1024], dtype=np.int64)).to(device)
        mask = torch.zeros((len(fields), 10), dtype=torch.bool, device=device)
        yield model(fields, mask, residual_only=True).flatten(1).cpu().numpy()